## Dependencies & Environment Setup

### Imports

In [ ]:
import os
import re
import sys
import string
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

try:
    import fasttext
except Exception:
    fasttext = None

### Project Root Detection

In [ ]:
# Find the project root by walking upward until requirements.txt is found
_project_root = Path.cwd().resolve()
for parent in [_project_root, *_project_root.parents]:
    if (parent / "requirements.txt").exists():
        _project_root = parent
        break
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))


### Path Constants

In [ ]:
PROJECT_ROOT = _project_root
DATASET_DIR = PROJECT_ROOT / "data" / "raw"
RESULT_DIR = PROJECT_ROOT / "results"
TRANSLATED_CSV_DIR = RESULT_DIR / "translated"
CLEANED_CSV_DIR = PROJECT_ROOT / "data" / "interim" / "cleaned"
FASTTEXT_MODEL_DIR = PROJECT_ROOT / "models" / "fasttext"
FASTTEXT_MODEL_NAME = "lid.176.bin"
FASTTEXT_MODEL_PATH = FASTTEXT_MODEL_DIR / FASTTEXT_MODEL_NAME

# Raw per-app review csv locations
DATASET_PATH_DICT = {
    "bKash": DATASET_DIR / "bkash_bd_reviews.csv",
    "Nagad": DATASET_DIR / "nagad_bd_reviews.csv",
    "Rocket": DATASET_DIR / "rocket_bd_reviews.csv",
}

REQUIRED_RAW_CSV_COLUMNS = [
    "app_name", "review_id", "review_date", "rating", "review_text",
    "thumbs_up_count", "app_version"
]
REQUIRED_MASTER_CSV_COLUMNS = [
    "app_name", "review_id", "review_date", "rating", "review_text",
    "review_text_clean", "thumbs_up_count", "app_version"
]
REQUIRED_SCORED_MASTER_CSV_COLUMNS = REQUIRED_MASTER_CSV_COLUMNS + [
    "sentiment_score", "gap_score", "mismatch_label"
]

### Path Related Helpers

In [ ]:
def get_run_paths(only_english: bool = False) -> dict:
    """Return output directories for the current run mode."""
    mode_name = "only_english" if only_english else "bangla_english"
    result_dir = PROJECT_ROOT / "results" / mode_name
    translated_dir = result_dir / "translated"
    cleaned_dir = PROJECT_ROOT / "data" / "interim" / "cleaned" / mode_name
    master_csv_path = result_dir / "master_bd_reviews.csv"
    scored_master_csv_path = result_dir / "master_bd_reviews_scored.csv"
    return {
        "mode_name": mode_name,
        "result_dir": result_dir,
        "translated_dir": translated_dir,
        "cleaned_dir": cleaned_dir,
        "master_csv_path": master_csv_path,
        "scored_master_csv_path": scored_master_csv_path,
    }


def ensure_dirs(only_english: bool = False):
    """Create the output folders used by this notebook, without relying on project modules."""
    run_paths = get_run_paths(only_english=only_english)
    run_paths["result_dir"].mkdir(parents=True, exist_ok=True)
    run_paths["translated_dir"].mkdir(parents=True, exist_ok=True)
    run_paths["cleaned_dir"].mkdir(parents=True, exist_ok=True)
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    TRANSLATED_CSV_DIR.mkdir(parents=True, exist_ok=True)
    CLEANED_CSV_DIR.mkdir(parents=True, exist_ok=True)
    FASTTEXT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
ONLY_ENGLISH = False
ensure_dirs(only_english=ONLY_ENGLISH)
run_paths = get_run_paths(only_english=ONLY_ENGLISH)

## Dictionaries & Normalization Config

In [ ]:
# Shared dictionaries for debugging clarity
PRE_TRANSLATION_DICT = {
    'aap': 'app', 'absulatlly': 'absolutely', 'acaunt': 'account', 'ae': 'app',
    'agant': 'agent', 'akti': 'a', 'amezing': 'amazing', 'ami': 'i', 'amr': 'my',
    'amy': 'i', 'aop': 'app', 'ap': 'app', 'apc': 'app', 'apk': 'app', 'apos': 'apps',
    'appa': 'apps', 'appd': 'app', 'appe': 'app', 'appy': 'app', 'aps': 'app',
    'apss': 'app', 'argent': 'urgent', 'arzent': 'urgent', 'ata': 'this', 'ato': 'very',
    'balo': 'good', 'bari': 'very', 'batter': 'better', 'beawtifull': 'beautiful',
    'bebohar kore': 'use', 'beri': 'very', 'bhalo': 'good', 'bikas': 'bkash',
    'bikash': 'bkash', 'bks': 'bkash', 'bksh': 'bkash', 'bonas': 'bonus',
    'bossor': 'year', 'bs': 'bullshit', 'chai': 'want', 'chalu': 'start',
    'cod': 'code', 'commisiion': 'commission', 'den na': 'does not give',
    'dey na': 'does not give', 'dipertment': 'department', 'distap': 'disturb',
    'doog': 'good', 'eazy': 'easy', 'ebong': 'and', 'et': 'it', 'experence': 'experience',
    'f': 'app', 'faltu': 'useless', 'famelly': 'family', 'faul': 'useless',
    'fiks': 'fix', 'forbikas': 'for bkash', 'fst': 'fast', 'full fill': 'fulfil',
    'g': 'good', 'gd': 'good', 'god': 'good', 'godd': 'good', 'goid': 'good',
    'goo': 'good', 'goob': 'good', 'good serve': 'good service', 'goods': 'good',
    'goody': 'good', 'goof': 'good', 'goog': 'good', 'goot': 'good', 'greet': 'great',
    'gud': 'good', 'hamfull': 'harmful', 'help ful': 'helpful', 'help full': 'helpful',
    'helped full': 'helpful', 'helpfuul': 'helpful', 'helpul': 'helpful', 'hepi': 'happy',
    'hotasa jonok': 'disappointing', 'ken': 'why', 'khob': 'very', 'khub': 'very',
    'khuv': 'very', 'kintu': 'but', 'korte parci na': 'i can not', 'korte parina': 'i can not',
    'korun': 'do', 'kurte paachhe ne': 'can not do', 'lakte se': 'feel', 'leading': 'loading',
    'lenden': 'transactions', 'lendener jonno': 'for transations', 'lon': 'loan',
    'lone': 'loan', 'loun': 'loan', 'lov': 'love', 'lugin': 'login', 'mi': 'my',
    'mobil': 'mobile', 'na': 'no/not', 'nace': 'nice', 'nacs': 'nice', 'nagod': 'nagad',
    'nah': 'no/not', 'naic': 'nice', 'naice': 'nice', 'naiec': 'nice', 'naies': 'nice',
    'nais': 'nice', 'nar bar': 'everytime', 'nc': 'nice', 'neis': 'nice', 'netay': 'get',
    'nic': 'nice', 'niceapp': 'nice app', 'nicee': 'nice', 'nices': 'nice', 'nich': 'nice',
    'nicr': 'nice', 'nieas': 'nice', 'niec': 'nice', 'nihc': 'nice', 'nise': 'nice',
    'nive': 'nice', 'noce': 'nice', 'nuch': 'nice', 'onak': 'very', 'onek': 'very',
    'onek din': 'many days', 'onk': 'very', 'outhe': 'other', 'paina': "can't get",
    'parmeshon': 'permission', 'parson': 'person', 'pls': 'please', 'por': 'after',
    'prblm': 'problem', 'rael': 'real', 'sait': 'site', 'sand': 'send', 'sapot': 'support',
    'sar': 'sir', 'scame': 'scam', 'seaport': 'support', 'servar': 'server', 'sheba': 'service',
    'shlw': 'slow', 'slove': 'solve', 'solow': 'slow', 'sonetimes': 'sometimes', 'soo': 'so',
    'sudi': 'bad', 'sundor': 'nice', 'supar': 'super', 'suppar': 'super', 'supper': 'super',
    'suprot': 'support', 'survice': 'service', 'taki': 'thaki', 'teke': 'for', 'theif': 'thief',
    'thier': 'there', 'thku': 'thank you', 'thx': 'thanks', 'tik': 'fix', 'tnx': 'thanks',
    'transection': 'transaction', 'trast': 'trust', 'use full': 'useful', 'use less': 'useless',
    'v.goods': 'very good', 'vai': 'brother', 'valo': 'good', 'valobasa': 'love', 'vare': 'very',
    'varey': 'very', 'vary': 'very', 'veery': 'very', 'veri': 'very', 'verry': 'very',
    'very bed': 'very bad', 'very safety': 'very safe', 'vhalo': 'good', 'vi': 'brother',
    'via': 'brother', 'vire': 'very', 'viry': 'very', 'weel': 'well', 'welcum': 'welcome',
    'whay': 'why', 'wonder full': 'wonderful',
    'আপ': 'app', 'আবরেট': 'update', 'এফ': 'app', 'ওক': 'ok', 'ওটিবি': 'otp',
    'কেন': 'why', 'গুট': 'good', 'চুমুতকা': 'nice', 'জামেলা': 'hassle', 'টাকা': 'bdt',
    'নাইচ': 'nice', 'নাইছ': 'nice', 'নাইস': 'nice', 'নাববার': 'number', 'বাঝে': 'bad',
    'বালের': 'useless', 'বালের এপ': 'useless app', 'বালো': 'good', 'বেড': 'bad',
    'ভালো': 'good', 'লেন দেন': 'transactions', 'সুদর': 'beautiful',
}

In [ ]:
POST_TRANSLATION_DICT = {"naicee": "nice", "nicee": "nice", "woww": "wow"}

In [ ]:
APP_SPECIFIC_TRANSLATION_DICT = {
    "bKash": {"বিকাশ": "bKash", "bikas": "bKash", "bikash": "bkash", "vikas": "bKash", "vikash": "bKash"},
    "Rocket": {},
    "Nagad": {"নগদ": "nagad", "nagod": "nagad", "nogod": "nagad", "নগদে": "in nagad app", "nogot": "nagad", "nagadel": "nagad's"},
}

In [ ]:
BANGLISH_KEYWORDS = {"ache", "amader", "apnader", "bhalo", "beshi", "boro", "choto", "dao", "faltu", "hoy", "kharap", "jabo", "khub", "korbo", "kisu", "lagbe", "nao", "nai", "onek", "pabo", "taka", "tader", "sundor", "theke", "valo"}

## Initializing Language Models

In [ ]:
def ensure_nltk_data():
    """Download the NLTK resources needed by the notebook's text-processing steps."""
    import nltk
    for resource in ("punkt", "punkt_tab", "stopwords", "wordnet"):
        try:
            nltk.data.find(resource)
        except LookupError:
            nltk.download(resource, quiet=True)


def ensure_fasttext_model():
    """Download the FastText language-identification model if it is not already present."""
    FASTTEXT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    if not FASTTEXT_MODEL_PATH.exists():
        import urllib.request
        print("Downloading FastText language model...")
        urllib.request.urlretrieve(
            "https://dl.fbaipublicfiles.com/fasttext/supervised-models/" + FASTTEXT_MODEL_NAME,
            FASTTEXT_MODEL_PATH,
        )
        print("Download complete.")


def load_fasttext_model():
    """Load the cached FastText model for language detection."""
    if fasttext is None:
        raise RuntimeError("fasttext is not installed")
    ensure_fasttext_model()
    return fasttext.load_model(str(FASTTEXT_MODEL_PATH))

In [ ]:
ensure_nltk_data()
ensure_fasttext_model()

## Text Preprocesssing Utility Functions

In [ ]:
def apply_custom_dict(text: str, mapping: dict) -> str:
    """Apply a token-level replacement dictionary to normalize noisy review text."""
    for key, value in mapping.items():
        text = re.sub(rf"\b{re.escape(key)}\b", value, text)
    return text


def convert_emojis(text: str) -> str:
    """Normalize emoji text without changing the rest of the review content."""
    return text


def detect_language(text: str):
    """Detect the review language and return the code plus a confidence value."""
    try:
        lang = detect(text)
        conf = 0.95
        return lang, conf
    except LangDetectException:
        return "und", 0.0


def translate_to_english(text: str, detected_lang: str, confidence: float, only_english: bool = False) -> str:
    """Translate non-English review text to English unless the notebook is in English-only mode."""
    if only_english and detected_lang != "en":
        return text
    if detected_lang == "en":
        return text
    try:
        return GoogleTranslator(source="auto", target="en").translate(text)
    except Exception:
        return text


def clean_text(text: str) -> str:
    """Normalize review text by lowercasing, stripping URLs, and removing punctuation."""
    text = text.lower()
    text = re.sub(r"https?\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def score_sentiment(text: str) -> float:
    """Return the VADER compound sentiment score for a cleaned review string."""
    sentiment = SentimentIntensityAnalyzer().polarity_scores(text)
    return sentiment["compound"]

## The Pipeline

In [ ]:
def standardize_review(text: str, app_name: str, only_english: bool = False):
    """Run the notebook's full text-normalization pipeline for one review record."""
    text = text.lower()
    pre_text = apply_custom_dict(text, PRE_TRANSLATION_DICT | APP_SPECIFIC_TRANSLATION_DICT.get(app_name, {}))
    emoji_converted = convert_emojis(pre_text)
    detected_lang, confidence = detect_language(emoji_converted)
    translated_text = translate_to_english(emoji_converted, detected_lang, confidence, only_english=only_english)
    cleaned_text = clean_text(translated_text)
    post_text = apply_custom_dict(cleaned_text, POST_TRANSLATION_DICT)
    return {
        "original_text": text,
        "pre_translated_text": pre_text,
        "emoji_converted": emoji_converted,
        "detected_lang": detected_lang,
        "confidence": round(float(confidence), 3),
        "translated_text": translated_text,
        "cleaned_text": cleaned_text,
        "post_translated": post_text,
    }


def _to_numeric_rating(rating):
    """Safely coerce a rating value to float, returning NaN for invalid inputs."""
    try:
        return float(rating)
    except Exception:
        return np.nan


def build_master_csv(only_english: bool = False):
    """Build a cleaned master CSV for the selected run mode and save it to disk."""
    frames = []
    for app_name, raw_path in DATASET_PATH_DICT.items():
        df = pd.read_csv(raw_path)
        df = df[REQUIRED_RAW_CSV_COLUMNS].copy()
        df = df.dropna(subset=["review_text"]).reset_index(drop=True)
        df["app_version"] = df["app_version"].fillna("unknown")

        step_results = df.apply(
            lambda row: standardize_review(row["review_text"], row["app_name"], only_english=only_english),
            axis=1,
        )
        steps_df = pd.DataFrame(step_results.tolist())
        steps_df.insert(0, "review_id", df["review_id"].values.tolist())

        if only_english:
            mask = (steps_df["detected_lang"] == "en").to_numpy()
            df = df[mask].reset_index(drop=True)
            steps_df = steps_df[mask].reset_index(drop=True)

        base = Path(raw_path).stem
        translated_path = run_paths["translated_dir"] / f"{base}_translated.csv"
        steps_df.to_csv(translated_path, index=False)
        df.insert(df.columns.get_loc("review_text") + 1, "review_text_clean", steps_df["post_translated"].values)
        frames.append(df)

    master_df = pd.concat(frames, ignore_index=True)
    output_path = run_paths["master_csv_path"]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    master_df.to_csv(output_path, index=False)
    return master_df


def load_master_csv(path, required_columns=REQUIRED_SCORED_MASTER_CSV_COLUMNS):
    """Load a CSV and raise if the required output columns are missing."""
    df = pd.read_csv(path)
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    return df


def score_master_csv(master_csv_path=None, only_english: bool = False):
    """Add sentiment, gap, and mismatch classification columns to the master dataset."""
    if master_csv_path is None:
        master_csv_path = run_paths["master_csv_path"]
    df = pd.read_csv(master_csv_path)
    df = df.copy()
    df["sentiment_score"] = df["review_text_clean"].fillna("").apply(score_sentiment)
    df["gap_score"] = df["rating"].astype(float) - df["sentiment_score"]
    df["mismatch_label"] = np.where(df["gap_score"] > 0, "Inflated", "Deflated")
    scored_path = run_paths["scored_master_csv_path"]
    df.to_csv(scored_path, index=False)
    return df


## Analysis & Reporting

In [ ]:
def run_full_analysis(master_csv_path, verbose: bool = True, lda_per_app: bool = True):
    """Compute summary statistics and trend output for the scored master dataframe."""
    df = load_master_csv(master_csv_path)
    summary_df = df.groupby("app_name").agg(
        reviews=("review_id", "count"),
        mean_rating=("rating", "mean"),
        mean_sentiment=("sentiment_score", "mean"),
        mean_gap=("gap_score", "mean"),
    ).reset_index()
    summary_df.to_csv(Path(master_csv_path).resolve().parent / "app_level_summary.csv", index=False)

    trend_df = df.assign(month=pd.to_datetime(df["review_date"]).dt.to_period("M").astype(str)).groupby(["month", "app_name"], as_index=False)["gap_score"].mean()
    trend_df.to_csv(Path(master_csv_path).resolve().parent / "monthly_gap_trend.csv", index=False)

    if verbose:
        print(summary_df.head())
        print(trend_df.head())

    return summary_df, trend_df, {}

## Visualization Helpers

In [ ]:
def plot_rating_vs_sentiment(df: pd.DataFrame, save_path: str | None = None) -> None:
    """Scatter-plot rating against sentiment score to inspect rating sentiment mismatches."""
    plt.figure(figsize=(8, 6))
    plt.scatter(df["rating"], df["sentiment_score"], alpha=0.6)
    plt.xlabel("Rating")
    plt.ylabel("Sentiment score")
    plt.title("Rating vs sentiment")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

In [ ]:
def plot_mismatch_composition(summary_df: pd.DataFrame, save_path: str | None = None) -> None:
    """Bar plot showing the mean gap score by app for mismatch composition analysis."""
    plt.figure(figsize=(8, 6))
    summary_df.set_index("app_name")["mean_gap"].plot(kind="bar")
    plt.title("Mean gap by app")
    plt.ylabel("Mean gap score")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

In [ ]:
def plot_monthly_gap_heatmap(trend_df: pd.DataFrame, save_path: str | None = None) -> None:
    """Heatmap of mean gap score across apps and months for trend inspection."""
    pivot = trend_df.pivot(index="app_name", columns="month", values="gap_score")
    plt.figure(figsize=(10, 4))
    sns.heatmap(pivot, cmap="coolwarm")
    plt.title("Monthly gap heatmap")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

In [ ]:
def plot_lda_wordclouds(lda_results: dict, save_path: str | None = None) -> None:
    """Create a simple word-cloud view from the LDA topic dictionary for debugging."""
    plt.figure(figsize=(8, 8))
    wc = WordCloud(width=600, height=400, background_color="white")
    wc.generate(" ".join(str(key) for key in lda_results.keys()))
    plt.imshow(wc)
    plt.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

In [ ]:
def plot_monthly_gap_trend(trend_df: pd.DataFrame, save_path: str | None = None) -> None:
    """Plot monthly gap score trends for each app as a multi-line time series."""
    plt.figure(figsize=(8, 5))
    for app_name, group in trend_df.groupby("app_name"):
        plt.plot(group["month"], group["gap_score"], label=app_name, marker="o")
    plt.title("Monthly gap trend")
    plt.xlabel("Month")
    plt.ylabel("Mean gap score")
    plt.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

## Execution Block

#### Prepare Paths & Load or Build Data

In [ ]:
ONLY_ENGLISH = False  # set True to keep only English-language reviews and skip translation
ensure_dirs(only_english=ONLY_ENGLISH)
run_paths = get_run_paths(only_english=ONLY_ENGLISH)

# Step 1: build or reuse the unscored master csv of cleaned reviews
master_df = None
if run_paths["master_csv_path"].exists():
    master_df = load_master_csv(
        run_paths["master_csv_path"],
        required_columns=REQUIRED_MASTER_CSV_COLUMNS,
    )
else:
    master_df = build_master_csv(only_english=ONLY_ENGLISH)

if master_df is None:
    raise ValueError("master_df is None")

# Step 2: score the master csv (sentiment + gap scores)
scored_master_df = None
if run_paths["scored_master_csv_path"].exists():
    scored_master_df = load_master_csv(run_paths["scored_master_csv_path"])
else:
    scored_master_df = score_master_csv(
        run_paths["master_csv_path"],
        only_english=ONLY_ENGLISH,
    )

if scored_master_df is None:
    raise ValueError("scored_master_df is None")

In [ ]:
del master_df

#### Analysis

In [ ]:
ensure_nltk_data()

In [ ]:
# Run full analysis: app-level stats, monthly trend, LDA topics per app
result = run_full_analysis(
    run_paths["scored_master_csv_path"],
    verbose=True,
    lda_per_app=True,
)

if result is not None:
    summary_df, trend_df, lda_results = result
    del result

#### Visualization

In [ ]:
if scored_master_df is not None:
    plot_rating_vs_sentiment(scored_master_df, save_path=str(run_paths["result_dir"] / "scatter_rating_vs_sentiment.png"))

In [ ]:
if summary_df is not None:
    plot_mismatch_composition(summary_df, save_path=str(run_paths["result_dir"] / "stacked_bar_mismatch.png"))

In [ ]:
if trend_df is not None:
    plot_monthly_gap_heatmap(trend_df, save_path=str(run_paths["result_dir"] / "monthly_gap_heatmap.png"))

In [ ]:
if lda_results is not None:
    plot_lda_wordclouds(lda_results, save_path=str(run_paths["result_dir"] / "wordclouds.png"))

In [ ]:
if trend_df is not None:
    plot_monthly_gap_trend(trend_df, save_path=str(run_paths["result_dir"] / "monthly_gap_line_chart.png"))